# 0.5.3 - XGBoost Building Evaluation

The 0.5.1 classification with one row per building: a building exceeds if any of its measurements is ≥ 300 Bq/m3, and is flagged if any of its predictions is above the threshold. Thresholds are the ones calibrated on measurements in 0.5.1 (atmos predicts for one room at a time), so these results describe what happens when every measured room of a building is predicted.

## Setup

In [ ]:
import json
import pickle

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import classification_report

from evaluation import (
    RADON_MAP_COLUMN,
    bootstrap_ci,
    by_building,
    classification_scores,
    load_inhabited,
    load_radon_map,
    load_switzerland,
    load_test_egid,
    load_test_set,
    operating_points,
    plot_building_maps,
    plot_calibrated_predicted_vs_observed,
    plot_confusion_matrix,
    plot_precision_recall,
    radon_map_comparison,
)

ROOT = '../../'
DIR = '../../models/'

## Data and model

In [ ]:
X_test, y_test = load_test_set(ROOT)
with open(f'{DIR}0.2-encoding-pipeline.pkl', 'rb') as file:
    preprocessor = pickle.load(file)

model = xgb.XGBRegressor()
model.load_model(f'{DIR}0.5-xgboost-log-model.json')
y_pred = np.expm1(model.predict(preprocessor.transform(X_test)))

# Thresholds calibrated on measurements in 0.5.1
with open(f'{DIR}0.5-decision-threshold.json') as file:
    settings = json.load(file)
RADON_REFERENCE_LEVEL, DECISION_THRESHOLD = settings['reference_level'], settings['decision_threshold']
THRESHOLDS = {'Decision': DECISION_THRESHOLD, **{f'Map {cutoff}': threshold for cutoff, threshold in settings['map_matched_thresholds'].items()}}

test = pd.DataFrame({'observed': y_test, 'predicted': y_pred, 'radon_map': X_test[RADON_MAP_COLUMN],
                     'east': X_test['Coordonnée E'], 'north': X_test['Coordonnée N']})
egid_test = load_test_egid(ROOT)
buildings = by_building(test, egid_test)
exceeds = buildings['observed'] >= RADON_REFERENCE_LEVEL
len(test), len(buildings)

## Classification

In [ ]:
operating_points(exceeds, buildings['predicted'], THRESHOLDS)

In [ ]:
plot_precision_recall(exceeds, buildings['predicted'], 'Precision–recall, test buildings - XGBoost', path=f'{ROOT}reports/figure-precision-recall-buildings')

In [ ]:
flagged = buildings['predicted'] >= DECISION_THRESHOLD
labels = [f'< {RADON_REFERENCE_LEVEL}', f'≥ {RADON_REFERENCE_LEVEL}']
print(classification_report(exceeds, flagged, target_names=labels))
plot_confusion_matrix(exceeds, flagged, f'Buildings, decision threshold {DECISION_THRESHOLD:.0f} Bq/m3 - XGBoost', RADON_REFERENCE_LEVEL)

In [ ]:
plot_calibrated_predicted_vs_observed(buildings['observed'], buildings['predicted'], DECISION_THRESHOLD, RADON_REFERENCE_LEVEL, 'Calibrated predictions, test buildings - XGBoost')

### Confidence intervals

Bootstrap over buildings (1,000 samples).

In [ ]:
bootstrap_ci(lambda t, p: classification_scores(t, p, DECISION_THRESHOLD, RADON_REFERENCE_LEVEL), buildings['observed'], buildings['predicted'], buildings.index)

## Comparison with the FOPH radon map

The radon map gives the local probability that a house exceeds 300 Bq/m3, where each house is represented by its highest measurement **in inhabited rooms**, the legally binding value (Kropat et al., 2014, J. Environ. Radioact. 129, 7–22). For a fair comparison, both are evaluated on the same quantity: test buildings, inhabited rooms only, highest observed and highest predicted concentration per building. Buildings without a measurement in an inhabited room or without a map value are excluded.

The map legend groups the probability into four classes; the boundaries between them (1 %, 10 %, 20 %) define three screening rules, "measure every building above the boundary":

| Boundary | Map classes flagged | Screening policy |
|---|---|---|
| > 1 % | 2–10 %, 11–20 %, > 20 % | Broad: everything except the lowest class |
| > 10 % | 11–20 %, > 20 % | Targeted |
| > 20 % | > 20 % | High-priority areas only |

For each rule, the model threshold is set on validation buildings (same definition) to find the same share of exceeding buildings as the map (equal recall); both are then scored on test buildings. At equal recall, a lower share of flagged buildings means fewer measurements for the same detection.

In [ ]:
RADON_MAP_CUTOFFS = [0.01, 0.10, 0.20]  # FOPH legend class boundaries

val = pd.read_csv(f'{DIR}0.5-xgboost-val-predictions.csv', index_col=0)
val['radon_map'] = load_radon_map(ROOT, val.index)
egid_val = pd.read_csv(f'{ROOT}data/final/dataset-v0.2-egid-train.csv', index_col=0)['EGID'].loc[val.index]

# Inhabited rooms only, highest value per building, as in the radon map
inhabited_test, inhabited_val = test[load_inhabited(ROOT, test.index)], val[load_inhabited(ROOT, val.index)]
inhabited_buildings = by_building(inhabited_test, egid_test.loc[inhabited_test.index])
inhabited_val_buildings = by_building(inhabited_val, egid_val.loc[inhabited_val.index])

comparison = radon_map_comparison(inhabited_buildings, inhabited_val_buildings, RADON_MAP_CUTOFFS, RADON_REFERENCE_LEVEL)
comparison

In [ ]:
covered = inhabited_buildings.dropna(subset=['radon_map'])
plot_precision_recall(covered['observed'] >= RADON_REFERENCE_LEVEL, covered['predicted'], 'Model vs radon map, test buildings - XGBoost',
                      path=f'{ROOT}reports/figure-precision-recall-radon-map', map_points=comparison.query("Method == 'Radon map'"))

### Maps

The same buildings on their coordinates: highest observed and highest predicted concentration in inhabited rooms (same colour scale), next to the radon map. The radon map gives a probability of exceeding 300 Bq/m3, not a concentration, so it is compared by pattern rather than by value.

The measurements are confidential: each building is moved by a random 100–200 m in a random direction before plotting (the same displacement in the three panels).

In [ ]:
plot_building_maps(covered, RADON_REFERENCE_LEVEL, path=f'{ROOT}reports/figure-maps-buildings', boundaries=load_switzerland(ROOT))